In [1]:
# 1. import dependencies

In [2]:
import torch

In [3]:
from guppylm.inference import GuppyInference

In [4]:
# 2. create the engine that we'll use for tokenization
# and generaton

In [5]:
engine = GuppyInference("checkpoints/best_model.pt", "data/tokenizer.json", "cpu")

GuppyLM loaded: 8.7M params


In [6]:
# this newly-creted engine has a model:

In [7]:
engine.model

GuppyLM(
  (tok_emb): Embedding(4096, 384)
  (pos_emb): Embedding(128, 384)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (out): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): FFN(
        (up): Linear(in_features=384, out_features=768, bias=True)
        (down): Linear(in_features=768, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
  (lm_head): Linear(in_features=384, out_features=4096, bias=False)
)

In [8]:
# now create an input line:

In [9]:
messages = [{"role": "user", "content": "hi"}]

In [10]:
prompt = engine._format_prompt(messages)

In [11]:
# this prompt is encoded as tokens.
# https://github.com/huggingface/tokenizers documents how this
# tokenization works.

In [12]:
input_ids = engine.tokenizer.encode(prompt).ids

In [13]:
input_ids

[1, 70, 46, 755, 2, 46, 1, 71, 46]

In [14]:
# turn that string of tokens into a tensor (whatever a tensor is)

In [15]:
input_t = torch.tensor([input_ids],dtype=torch.long,device=engine.device)

In [16]:
input_t

tensor([[  1,  70,  46, 755,   2,  46,   1,  71,  46]])

In [17]:
# then generate a response based on that tensor

In [18]:
output_t,_ = engine.model.generate(input_t,64,0.7,50)

In [19]:
output_t

tensor([[  1,  70,  46, 755,   2,  46,   1,  71,  46, 634, 843,   5,  77, 217,
          93, 265, 104, 470, 248, 353, 119,   5,   2]])

In [20]:
# .. and finally convert that human-readable form:

In [21]:
engine.tokenizer.decode(output_t[0].tolist())

'user\nhi\nassistant\noh hi. the light is nice to say so maybe that.'